# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, leveraging the dataset's Croissant JSON-LD schema and following best practices for interacting with complex schemas using the `@id` field for all references.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and access dataset structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL (as provided)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('Version:', getattr(metadata, 'version', None))
print('Date Published:', getattr(metadata, 'datePublished', None))


## 2. Data Overview
List all available record sets in the dataset, referencing each by its `@id`. For each record set, display its fields and their `@id`s.

In [ ]:
print('Record Sets Available:')
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        # rs is a RecordSetMetadata
        print(f"- recordSet @id: {rs['@id']} | name: {rs.get('name', '(no name)')}")
        record_sets.append(rs['@id'])
        # List the fields for this record set
        field_ids = rs.get('field', [])
        print('  Fields (by @id):')
        for fld in field_ids:
            if isinstance(fld, dict):
                print(f"    - {fld.get('@id')}")
            else:
                print(f"    - {fld}")
else:
    print('No record sets specified in metadata!')

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. All entities (record sets, fields) are referenced by their Croissant `@id`.

In [ ]:
# Usually, we extract the main dataset. 
# Here, we programmatically get all record sets and extract.
dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        # Each record is a dict keyed by field `@id`
        print(f"Loading records for RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Fields/columns available: {list(dataframes[record_set_id].columns)}\n")
        else:
            print(f"No records found for RecordSet @id: {record_set_id}\n")
    # Pick the first one (if any) for EDA
    selected_recordset_id = record_sets[0]
    print(f"Sample records from RecordSet @id: {selected_recordset_id}")
    if selected_recordset_id in dataframes:
        display(dataframes[selected_recordset_id].head())
else:
    print('No record sets detected. Please check the dataset.')

## 4. Exploratory Data Analysis (EDA)
Below, we demonstrate filtering, normalization, and (if possible) grouping for a numeric field in the first detected record set. All references to fields are done using their `@id`.

Note: Adjust field `@id` as available in your specific dataset.

In [ ]:
# Find a numeric field from the DataFrame's columns using a heuristic
df = None
if record_sets and dataframes:
    selected_rs = record_sets[0]
    df = dataframes[selected_rs]
    # Try to select the first numeric column
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try to convert any column that looks like numbers to float
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field (by @id): {numeric_field_id}\n")
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 10
        # Filter
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:\n{filtered_df.head()}\n")
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:\n{filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head()}\n")
        # Try grouping by the next available field (if possible)
        other_fields = [col for col in df.columns if col != numeric_field_id]
        group_field = None
        for col in other_fields:
            if pd.api.types.is_categorical_dtype(df[col]) or df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"Mean {numeric_field_id} grouped by {group_field} (@id):\n{grouped_df.head()}")
    else:
        print('No numeric fields detected in this record set.')
else:
    print('No record sets or dataframes available.')

## 5. Visualization
Let's plot the numeric field's distribution and, if grouped, show group means.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=60)
        plt.show()

## 6. Conclusion
In this notebook, we loaded a Croissant-based dataset with `mlcroissant`, explored available record sets and their fields by `@id`, extracted records, and carried out basic exploratory analysis and visualization using only provenance-preserving `@id` references. 

- All dataset components—record sets, fields—were referenced exclusively by their Croissant `@id`.
- Numeric and grouping fields can be programmatically discovered with this pattern for arbitrary Croissant datasets.
- For more advanced analysis, refer directly to field `@id` from section 2.